In [ ]:
import numpy as np
import xarray as xr
import sys
sys.path.append('../')
import functions.HelperFunctions as HelperFunctions
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
import functions.Dictionaries as Dictionaries
import importlib
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from windspharm.xarray import VectorWind
import cmocean
import pandas as pd
from matplotlib.animation import FuncAnimation, PillowWriter
from tqdm.notebook import tqdm
import glob

In [ ]:
importlib.reload(HelperFunctions)
years = np.arange(2010,2024+1)

months_array = np.arange(1,12+1)
days = np.arange(1,365+1)
months = [f"{m:02d}" for m in months_array]

maps_list = []

for year in years:
    monthly_datasets = []

    for m in months:
        filename = (
            f"/scratch/projects/phase01/rab/B06/PROCESSED_ERA5/"
            f"{year}/TurbulenceMapsTimesteps_{year}{m}.zarr"
        )

        ds = xr.open_zarr(filename)

        # Normalize longitude to [-180, 180)
        lon = ((ds.longitude + 180) % 360) - 180
        ds = ds.assign_coords(longitude=lon)

        # Remove duplicate longitudes, if present
        _, idx = np.unique(
            ds.longitude.values,
            return_index=True
        )
        if len(idx) != ds.sizes["longitude"]:
            ds = ds.isel(longitude=np.sort(idx))

        ds = ds.sortby("longitude")
        monthly_datasets.append(ds)

    ds_year = xr.concat(
        monthly_datasets,
        dim="time",
        join="exact"
    )

    dates = pd.date_range(
        start=f"{year}-01-01",
        periods=ds_year.sizes["time"],
        freq="D"
    )
    ds_year = ds_year.assign_coords(time=dates)

    maps_list.append(ds_year)

daily_maps = xr.concat(maps_list, dim="time", join="exact")
daily_maps = daily_maps.assign_coords(year=("time", daily_maps.time.dt.year.data))

maps_list = []

for year in years:
    monthly_datasets = []

    for m in months:
        filename = (
            f"/scratch/projects/phase01/rab/B06/PROCESSED_ERA5/"
            f"{year}/TurbulenceMaps_{year}{m}.zarr"
        )

        ds = xr.open_zarr(filename)

        # Normalize longitude to [-180, 180)
        lon = ((ds.longitude + 180) % 360) - 180
        ds = ds.assign_coords(longitude=lon)

        # Remove duplicate longitudes, if present
        _, idx = np.unique(
            ds.longitude.values,
            return_index=True
        )
        if len(idx) != ds.sizes["longitude"]:
            ds = ds.isel(longitude=np.sort(idx))

        ds = ds.sortby("longitude")
        monthly_datasets.append(ds)

    ds_year = xr.concat(
        monthly_datasets,
        dim="time",
        join="exact"
    )

    dates = pd.date_range(
        start=f"{year}-01-01",
        periods=ds_year.sizes["time"],
        freq="MS"
    )
    ds_year = ds_year.assign_coords(time=dates)

    maps_list.append(ds_year)

monthly_maps = xr.concat(maps_list, dim="time", join="exact")
monthly_maps = monthly_maps.assign_coords(year=("time", monthly_maps.time.dt.year.data))

years = np.arange(2015,2024+1)

fields_list = []

for year in years:

    path = f"/scratch/projects/phase01/rab/B06/PROCESSED_ERA5/{year}/"

    months_array = np.arange(1, 13)
    months = [f"{m:02d}" for m in months_array]

    filenames_fields = [
        path + f"MeanField_{year}{m}.zarr"
        for m in months
    ]

    ds = xr.open_mfdataset(
        filenames_fields,
        engine="zarr",
        combine="nested",
        concat_dim="time"
    )
    dates = pd.date_range(
        start=f"{year}-01-01",
        periods=ds.sizes["time"],
        freq="MS"
    )

    ds = ds.assign_coords(time=dates)

    #ds = ds.rename(time = 'month').assign_coords(month = months_array)

    fields_list.append(ds)

fields = xr.concat(
    fields_list,
    dim="time",
)
lrt_list = []

for year in years:

    path = f"/scratch/projects/phase01/rab/B06/PROCESSED_ERA5/{year}/"

    filenames_lrt = [
        path + f"z_lrt_{year}{m}.zarr"
        for m in months
    ]

    ds = xr.open_mfdataset(
        filenames_lrt,
        engine="zarr",
        combine="nested",
        concat_dim="time"
    )

    days = pd.to_datetime(ds.time.values, unit="D", origin=f"{year}-01-01")
    ds = ds.rename(time = 'day')
    ds = ds.assign_coords(day = days)
    #ds = ds.rename(time = 'month').assign_coords(month = months_array)

    lrt_list.append(ds)

lrt = xr.concat(
    lrt_list,
    dim="year",
)

In [ ]:
lon_slice = slice(60, 130)
maps = daily_maps

monsoon_region = (
    maps.sel(latitude=slice(10, -10), longitude=lon_slice)
    .mean(dim=["latitude", "longitude"])
)

# Exclude February 29 for a 365-day climatology
no_feb29 = ~(
    (monsoon_region.time.dt.month == 2) &
    (monsoon_region.time.dt.day == 29)
)
monsoon_no_feb29 = monsoon_region.where(no_feb29, drop=True)

signal_climatology = (
    monsoon_no_feb29.groupby("time.dayofyear")
    .mean("time")
    .rolling(dayofyear=30, center=True, min_periods=1)
    .mean() * 100
)

climatology_std = (
    monsoon_no_feb29.groupby("time.dayofyear")
    .std("time")
    .rolling(dayofyear=30, center=True, min_periods=1)
    .mean() * 100
)

common_time = pd.date_range("2017-01-01", "2017-12-31", freq="D")

fig, ax = plt.subplots(figsize=(10, 5))

for year in range(2015, 2025):
    signal = (
        monsoon_region.where(
            monsoon_region.time.dt.year == year, drop=True
        ) * 100
    )
    signal = signal.rolling(time=30, center=True, min_periods=1).mean()

    # Remove Feb 29 so every year has 365 days
    dates = pd.DatetimeIndex(signal.time.values)
    keep = ~((dates.month == 2) & (dates.day == 29))
    signal = signal.isel(time=np.where(keep)[0])

    if year == 2017:
        ax.plot(
            common_time,
            signal.Ri.values,
            color="black",
            linewidth=2.5,
            label="2017",
            zorder=3
        )
    else:
        ax.plot(
            common_time,
            signal.Ri.values,
            color="lightgray",
            linewidth=1,
            alpha=0.8,
            zorder=1
        )

# Climatology (unchanged)
doy = np.arange(1, 366)
clim = signal_climatology.sel(dayofyear=doy)
std = climatology_std.sel(dayofyear=doy)

ax.plot(
    common_time, clim.Ri.values,
    color="fuchsia", linewidth=2,
    label="Climatology", zorder=4
)

ax.plot(
    common_time, (clim.Ri - std.Ri).values,
    color="fuchsia", linewidth=1,
    linestyle=(0, (5, 10)), zorder=2
)

ax.plot(
    common_time, (clim.Ri + std.Ri).values,
    color="fuchsia", linewidth=1,
    linestyle=(0, (5, 10)), zorder=2
)

ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax.set_xlim(common_time[0], common_time[-1])
ax.set_ylim(0, 50)

ax.set_xlabel("")
ax.set_ylabel(r"% with Ri$\leq 1$", fontsize = 14)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", alpha=0.3)
# ax.legend(frameon=False)

fig.autofmt_xdate()
plt.tight_layout()

plt.savefig('../../Plots/PaperReady/Appendix2017Climatology.png', format = 'PNG', dpi = 300)

In [ ]:
u_diff = (
    fields.u.sel(level=55, latitude = slice(15,-15)).groupby("time.season").mean("time") -
    fields.u.sel(level=62, latitude = slice(15,-15)).groupby("time.season").mean("time")
)
v_diff = (
    fields.v.sel(level=55, latitude = slice(15,-15)).groupby("time.season").mean("time") -
    fields.v.sel(level=62, latitude = slice(15,-15)).groupby("time.season").mean("time")
)

z_diff = (fields.height.sel(level=55, latitude = slice(15,-15)).groupby("time.season").mean("time") -
        fields.u.sel(level=62, latitude = slice(15,-15)).groupby("time.season").mean("time"))

S = ((u_diff/z_diff)**2 + (v_diff/z_diff)**2)*1e7
S = S.compute()

In [ ]:
X = monthly_maps.Ri.groupby('time.month') - monthly_maps.Ri.groupby('time.month').mean()
X = X.sel(latitude = slice(10, -10)).mean(dim = 'latitude')#.sel(longitude = slice(60,130))
X = X - X.mean(dim = 'time')
C = np.cov(X, rowvar = False)

w,v = np.linalg.eig(C)
idx_sort = np.argsort(w)[::-1]
w,v = w[idx_sort],v[:,idx_sort]

explained_variance = w/w.sum()
print(explained_variance.real[:10]*100)
plt.plot(np.arange(len(explained_variance)), explained_variance)
plt.xlim(0,10)
plt.ylabel('Explained Variance')
plt.xlabel('EOF')

In [ ]:
# First EOF in physical units
eof1 = np.sqrt(w[0].real) * v[:, 0].real*100

lon = X.longitude.values

fig, ax = plt.subplots(figsize=(8, 4.5), dpi=150)

lim = np.max(np.abs(eof1))

sc = ax.plot(
    lon, eof1,
    color = 'blue',
)

ax.plot(lon, eof1, color='0.3', linewidth=0.8, alpha=0.5)
ax.axhline(0, color='black', linewidth=0.8)

ax.set_xlabel('Longitude (°)')
ax.set_ylabel(r'Ri $\leq 1$ occurrence frequency [%]')
ax.set_title(
    f'First EOF — {explained_variance[0].real * 100:.2f}% explained variance'
)

ax.grid(True, linestyle='--', alpha=0.25)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('../../Plots/PaperReady/AppendixEOF1.png', format = 'PNG', dpi = 300)

In [ ]:
seasons = ["DJF", "MAM", "JJA", "SON"]
nino = xr.open_dataset('nino34.long.anom.nc').sel(time = slice('2010-01-01T00:00:00.000000','2024-12-01T00:00:00.000000'))

PC1 = X.values@v[:,0].real
PC1_std = (PC1 - PC1.mean()) / PC1.std()
PC1_da = xr.DataArray(
PC1_std,
coords={"time": X.time},
dims=["time"]
)#.rolling(time = 5).mean()

fig, axes = plt.subplots(
    nrows=5,
    ncols=1,
    figsize=(10, 6),
    constrained_layout=True,
    gridspec_kw={"height_ratios": [1, 1, 1, 1, 1.3]},
    subplot_kw={"projection": ccrs.PlateCarree()}
)

# --------------------------------------------------
# First four panels: seasonal maps
# --------------------------------------------------
for i, (ax, season) in enumerate(zip(axes[:4], seasons)):

    subset = (
        monthly_maps
        .where(monthly_maps.time.dt.season == season, drop=True)
        .mean(dim="time")
        .sel(latitude=slice(15, -15))
        .Ri * 100
    )

    mask = subset > -0.000001

    pcm = ax.pcolormesh(
        subset.longitude,
        subset.latitude,
        subset.where(mask).transpose("latitude", "longitude"),
        transform=ccrs.PlateCarree(),
        cmap=cmocean.cm.rain,
        norm=Normalize(vmin=0, vmax=35),
        shading="auto",
        rasterized=True,
    )

    ax.coastlines(resolution="110m", linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.4)
    ax.add_feature(
        cfeature.LAND, facecolor="lightgray",
        edgecolor="none", zorder=0
    )
    ax.add_feature(
        cfeature.OCEAN, facecolor="white", zorder=0
    )

    ax.set_yticks([-15, 0, 15], crs=ccrs.PlateCarree())
    ax.set_yticklabels(["15", "0", "-15"])

    ax.set_xticks(
        np.arange(-180, 181, 30),
        crs=ccrs.PlateCarree()
    )

    if season == "DJF":
            ax.text(
            0, 1.05, "(a)",
            transform=ax.transAxes,
            fontsize=12,
            fontweight="bold",
            va="bottom",
            ha="left",
            clip_on=False
        )

    if season == "JJA":
        ax.set_ylabel("Latitude [°]", fontsize=12)
    else:
        ax.set_ylabel("")

    if season != "SON":
        ax.set_xticklabels([])
    else:
        ax.set_xlabel("Longitude [°]", fontsize=12)
    ax.text(
        0, 1.05, f"({chr(97 + i)})",
        transform=ax.transAxes,
        fontsize=12,
        fontweight="bold",
        va="bottom",
        ha="left",
        clip_on=False
    )

    u_season = S.sel(season=season)

    cs = ax.contour(
        u_season.longitude,
        u_season.latitude,
        u_season,
        levels=[0, 2, 4, 6, 8, 10],
        colors="black",
        linewidths=0.7,
        transform=ccrs.PlateCarree(),
    )

    ax.clabel(cs, inline=True, fontsize=8, fmt="%d")



ax_ts = fig.add_subplot(5, 1, 5)

lon_slice = slice(60, 130)

monsoon_region = (
    monthly_maps
    .sel(
        latitude=slice(10, -10),
        longitude=lon_slice
    )
    .mean(dim=["latitude", "longitude"])
    * 100
)

# monsoon_region = monsoon_region.rolling(
#     time=9, center=True
# ).mean()

line1, = ax_ts.plot(
    monsoon_region.time,
    monsoon_region.Ri,
    color="black",
    linewidth=1.5,
    label="Occurrence frequency"
)

ax_ts.set_ylabel("Ri < 1 [%]", color="black")
ax_ts.set_xlabel("Time")
ax_ts.grid(True, alpha=0.25)
ax_ts.set_ylim(0,30)

# Right y-axis: PC1 and Niño index
ax_ts2 = ax_ts.twinx()

# line2, = ax_ts2.plot(
#     X.time, PC1_da,
#     color="tab:red",
#     linewidth=1,
#     label="PC1"
# )

line3, = ax_ts2.plot(
    X.time, nino.value,
    color="tab:blue",
    linewidth=1,
    label="Niño index"
)

ax_ts2.set_ylabel("Niño 3.4 Index", color = "tab:blue")

ax_ts2.set_ylim(-3,3)
ax_ts2.tick_params(axis="y", colors="tab:blue")

# Combined legend
lines = [line1, line3]
labels = [line.get_label() for line in lines]

ax_ts.text(
    0, 1.05, "(e)",
    transform=ax_ts.transAxes,
    fontsize=12,
    fontweight="bold",
    va="bottom",
    ha="left",
    clip_on=False
)

# ax_ts.legend(
#     lines, labels,
#     loc="upper left",
#     ncol=3,
#     fontsize=9
# )

cbar = fig.colorbar(
    pcm,
    ax=axes[:4].tolist(),
    aspect=15,
    shrink=0.9,
    pad=0.02,
    extend="max"
)

cbar.set_label(
    r"Occurrence frequency of Ri < 1 [%]",
    rotation=90,
    labelpad=15,
    fontsize=14
)

plt.savefig(
"../../Plots/PaperReady/RiMap.png",
format="PNG",
dpi=300,
bbox_inches="tight"
)